In [ ]:
import sys
sys.path.insert(0, '/work/phd_ultrasounds/US_FiLMUNet')

import torch
from safetensors.torch import load_file
from nets.unet_attn import UNet2DAttn
from nets.segm_net import UNet2DFiLM


In [ ]:
model = UNet2DAttn(
    in_channels=3,
    num_classes=1,
    n_organs=10,
    size=16,
    depth=5,
    attn_start=0,
    use_attn=True,
    img_size=512,
    patch_size=8,
    emb_dim=768,
    n_heads=8,
    distill=False,
    use_dwt=False,
    wavelet='haar',
    use_shape=False,
    shape_res=64,
)

In [ ]:
model = UNet2DFiLM(
    in_channels=3,
    num_classes=1,
    # n_organs=len(organ_to_class_dict),
    n_organs=8,
    size=32,
    depth=5,
    film_start=0,
    use_film=True,
    distill=False
)

In [ ]:
CHECKPOINT = '/work/phd_ultrasounds/US_FiLMUNet/loggings/dd9c669e4884/checkpoint-8400/model.safetensors'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

state_dict = load_file(CHECKPOINT)
state_dict = {k: v for k, v in state_dict.items() if 'distill' not in k}
load_result = model.load_state_dict(state_dict, strict=False)
print(load_result)

model.eval()
model.to(DEVICE)
print(f"Model loaded on {DEVICE}")

In [ ]:
import sys
sys.path.insert(0, '/work/phd_ultrasounds/US_FiLMUNet')

import torch
from safetensors.torch import load_file
from nets.unet_attn import UNet2DAttn

CHECKPOINT = '/work/phd_ultrasounds/US_FiLMUNet/loggings/dd9c669e4884/checkpoint-8400/model.safetensors'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = UNet2DAttn(
    in_channels=3,
    num_classes=1,
    n_organs=10,
    size=16,
    depth=5,
    attn_start=0,
    use_attn=True,
    img_size=512,
    patch_size=8,
    emb_dim=768,
    n_heads=8,
    distill=False,
    use_dwt=False,
    wavelet='haar',
    use_shape=False,
    shape_res=64,
)

state_dict = load_file(CHECKPOINT)
state_dict = {k: v for k, v in state_dict.items() if 'distill' not in k}
load_result = model.load_state_dict(state_dict, strict=False)
print(load_result)

model.eval()
model.to(DEVICE)
print(f"Model loaded on {DEVICE}")

In [ ]:
from torch.utils.data import DataLoader
from data_classes.datasets import USdatasetOmni
from utils.paths import DATA_DIR
from utils.utils import get_sft_transforms

BATCH_SIZE = 8

train_dataset = USdatasetOmni(
    base_dir=DATA_DIR,
    split='train',
    transforms=get_sft_transforms(train=False),
    out_size=512,
    data_type='segmentation',
    ccl_crop=False,
    keep_aspect_ratio=True,
    self_norm=False,
    id_dropout=0.0,
)

test_dataset = USdatasetOmni(
    base_dir=DATA_DIR,
    split='val',
    transforms=get_sft_transforms(train=False),
    out_size=512,
    data_type='segmentation',
    ccl_crop=False,
    keep_aspect_ratio=True,
    self_norm=False,
    id_dropout=0.0,
)

train_dataset.items = train_dataset.items[:2]
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_dataset.items = test_dataset.items[:2]
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} images | Test: {len(test_dataset)} images")

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt


def collect_modulations(model, loader, device):
    """
    Run inference and collect compact FiLM summaries for each image.

    Returns:
        gamma_maxes: (N, L, Cmax) cpu tensor
        beta_maxes:  (N, L, Cmax) cpu tensor
        organ_ids:   (N,) cpu tensor
    """
    if not getattr(model, 'use_attn', False):
        raise ValueError('This helper expects an attention-modulated model.')

    gamma_maxes, beta_maxes, organ_ids = [], [], []
    model.eval()
    layer_configs = model._build_layer_configs()

    with torch.no_grad():
        for batch in tqdm(loader):
            pixel_values = batch['pixel_values'].to(device)
            batch_organ_id = batch['organ_id'].long().to(device)

            _, _, compact_modulations = model.shared_attn.compute_all_modulations(
                pixel_values,
                batch_organ_id,
                layer_configs,
                return_compact=True,
            )

            batch_gamma_max = torch.stack(
                [compact_modulations[layer_id][0] for layer_id, _ in layer_configs],
                dim=1,
            )
            batch_beta_max = torch.stack(
                [compact_modulations[layer_id][1] for layer_id, _ in layer_configs],
                dim=1,
            )

            gamma_maxes.append(batch_gamma_max.cpu())
            beta_maxes.append(batch_beta_max.cpu())
            organ_ids.append(batch_organ_id.cpu())

    gamma_maxes = torch.cat(gamma_maxes, dim=0)
    beta_maxes = torch.cat(beta_maxes, dim=0)
    organ_ids = torch.cat(organ_ids, dim=0)
    return gamma_maxes, beta_maxes, organ_ids


def compact_for_heatmap(modulations, reduction='mean'):
    """Collapse the image axis to get a (layers, compact_channels) heatmap."""
    if reduction == 'mean':
        return modulations.mean(dim=0)
    if reduction == 'abs_mean':
        return modulations.abs().mean(dim=0)
    raise ValueError(f'Unsupported reduction: {reduction}')


gamma_max_test, beta_max_test, organ_ids_test = collect_modulations(model, train_loader, DEVICE)
# gamma_heatmap = compact_for_heatmap(gamma_max_test, reduction='mean')
# beta_heatmap = compact_for_heatmap(beta_max_test, reduction='mean')

# print(f'Test gamma_max shape: {gamma_max_test.shape}')
# print(f'Test beta_max shape:  {beta_max_test.shape}')
# print(f'Gamma heatmap shape: {gamma_heatmap.shape}')
# print(f'Beta heatmap shape:  {beta_heatmap.shape}')

In [ ]:
gamma_heatmap = compact_for_heatmap(torch.nn.functional.normalize(gamma_max_test[1], p=2).unsqueeze(0), reduction='mean')
beta_heatmap = compact_for_heatmap(torch.nn.functional.normalize(beta_max_test[1], p=2).unsqueeze(0), reduction='mean')

In [ ]:
torch.nn.functional.max_pool1d(gamma_max_test[1], kernel_size=2).shape

In [ ]:
gamma_heatmap = compact_for_heatmap(gamma_max_test[0].unsqueeze(0), reduction='mean')
beta_heatmap = compact_for_heatmap(beta_max_test[0].unsqueeze(0), reduction='mean')

In [ ]:
gamma_heatmap.shape

In [ ]:
gamma_heatmap = torch.nn.functional.max_pool1d(gamma_max_test[0], kernel_size=6)
beta_heatmap = torch.nn.functional.max_pool1d(beta_max_test[0], kernel_size=6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

im0 = axes[0].imshow(gamma_heatmap.numpy(), aspect='auto', cmap='viridis')
axes[0].set_title('Gamma max heatmap')
axes[0].set_xlabel('Compact channel')
axes[0].set_ylabel('FiLM layer')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(beta_heatmap.numpy(), aspect='auto', cmap='magma')
axes[1].set_title('Beta max heatmap')
axes[1].set_xlabel('Compact channel')
fig.colorbar(im1, ax=axes[1])

layer_ticks = list(range(gamma_heatmap.shape[0]))
for ax in axes:
    ax.set_yticks(layer_ticks)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

im0 = axes[0].imshow(gamma_heatmap.numpy(), aspect='auto', cmap='viridis')
axes[0].set_title('Gamma max heatmap')
axes[0].set_xlabel('Compact channel')
axes[0].set_ylabel('FiLM layer')
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(beta_heatmap.numpy(), aspect='auto', cmap='magma')
axes[1].set_title('Beta max heatmap')
axes[1].set_xlabel('Compact channel')
fig.colorbar(im1, ax=axes[1])

layer_ticks = list(range(gamma_heatmap.shape[0]))
for ax in axes:
    ax.set_yticks(layer_ticks)

plt.tight_layout()
plt.show()